## 0. Импорт

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC 
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import joblib

## 1. Предварительная обработка
Прочитайте файл dayofweek.csv, который вы использовали в предыдущий день, и сохраните его в виде датафрейма.
Используя train_test_split параметры test_size=0.2, random_state=21получите X_train, y_train, X_test, y_test. Используйте дополнительный параметр stratify.

In [2]:
df = pd.read_csv(
    '../data/dayofweek.csv', 
    index_col=0,
    )
df.head()

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.692958,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.661055,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [3]:
X = df[df.drop('dayofweek', axis=1).columns]
y = df['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)


## Регуляризация Logreg
### а. Регуляризация по умолчанию
1. Обучите базовую модель, используя только параметры random_state=21, fit_intercept=False.
2. Для оценки точности модели используйте стратифицированную K-кратную (10) перекрестную проверку с разбиением выборки.

In [4]:
logreg = LogisticRegression(random_state=21, fit_intercept=False)
skf = StratifiedKFold(n_splits=10, random_state=21, shuffle=True)
valid_scores = []

for train_idx, valid_idx in skf.split(X, y):
    X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
    y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
    
    logreg.fit(X_train_fold, y_train_fold)

    train_acc = logreg.score(X_train_fold, y_train_fold)
    valid_acc = logreg.score(X_valid_fold, y_valid_fold)

    valid_scores.append(valid_acc)

    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')
print(f'Std is {np.std(valid_scores):.5f}')

train -  0.63546   |   valid -  0.65089
train -  0.65326   |   valid -  0.60947
train -  0.63942   |   valid -  0.63314
train -  0.63283   |   valid -  0.57988
train -  0.65590   |   valid -  0.57988
train -  0.64535   |   valid -  0.62130
train -  0.63834   |   valid -  0.60714
train -  0.63702   |   valid -  0.59524
train -  0.64295   |   valid -  0.68452
train -  0.63900   |   valid -  0.56548
Average accuracy on crossval is 0.61269
Std is 0.03441


### б. Оптимизация параметров регуляризации
В ячейках ниже попробуйте разные значения штрафа: none, l1, l2– вы также можете изменить значения решателя.

**пункт из интернета чтобы не появлялись предупреждения:**   
В sklearn 1.8+ параметр penalty заменён на l1_ratio и C:

* penalty='l2' теперь l1_ratio=0
* penalty='l1' теперь l1_ratio=1
* penalty=None теперь C=1e10 + l1_ratio=0

In [5]:
param_combinations = [
    {'C': 1e10, 'l1_ratio': 0, 'solver': 'lbfgs', 'label': 'No penalty'},
    {'C': 1.0, 'l1_ratio': 1, 'solver': 'saga', 'label': 'L1 (saga)'},
    {'C': 1.0, 'l1_ratio': 0, 'solver': 'lbfgs', 'label': 'L2 (lbfgs)'},
    {'C': 1.0, 'l1_ratio': 0, 'solver': 'saga', 'label': 'L2 (saga)'},
]
best_acc = 0
best_l1_ratio = ''
best_solver = ''
best_C = ''
best_combination = ''
for params in param_combinations:
        valid_scores_temp = []

        temp_logreg = LogisticRegression(
                        random_state=21,l1_ratio=params['l1_ratio'],
                        solver=params['solver'], C=params['C']
                        ,fit_intercept=False, max_iter=1000)
        for train_idx, valid_idx in skf.split(X, y):
            X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
            y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
            
            temp_logreg.fit(X_train_fold, y_train_fold)
        
            train_acc = temp_logreg.score(X_train_fold, y_train_fold)
            valid_acc = temp_logreg.score(X_valid_fold, y_valid_fold)
        
            valid_scores_temp.append(valid_acc)

        if np.mean(valid_scores_temp) > best_acc:
            best_acc = np.mean(valid_scores_temp)
            best_combination = params['label']
        
print(f"Best average accuracy on crossval is: {best_acc:.5f}")
print(f"Best combination is: {best_combination}")

Best average accuracy on crossval is: 0.63402
Best combination is: No penalty


## 3. SVM регуляризация

### а. Регуляризация по умолчанию
1. Обучите базовую модель, используя только параметры `probability=True`, `kernel='linear'`, `random_state=21`.

2. Используйте стратифицированную K-кратную перекрестную проверку с разбиением на `10` ячеек для оценки точности модели.

3. Формат результата кода, в котором вы обучали и оценивали базовую модель, должен быть аналогичен формату, полученному для logreg.

In [6]:
svm = SVC(probability=True, kernel='linear', random_state=21)

In [7]:
valid_scores = []

for train_idx, valid_idx in skf.split(X, y):
    X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
    y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
    
    svm.fit(X_train_fold, y_train_fold)

    train_acc = svm.score(X_train_fold, y_train_fold)
    valid_acc = svm.score(X_valid_fold, y_valid_fold)

    valid_scores.append(valid_acc)

    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')
print(f'Std is {np.std(valid_scores):.5f}')

train -  0.70138   |   valid -  0.71598
train -  0.69677   |   valid -  0.68639
train -  0.70402   |   valid -  0.71006
train -  0.69941   |   valid -  0.63905
train -  0.71127   |   valid -  0.62130
train -  0.70336   |   valid -  0.69822
train -  0.69038   |   valid -  0.67857


KeyboardInterrupt: 

### б. Оптимизация параметров регуляризации
В ячейках ниже попробуйте разные значения параметра C.

In [79]:
C_range = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
best_acc = 0
best_C = ''
for c in C_range:
        valid_scores_temp = []
        # сделал probability=False вместо True чтобы ускорить запуск (когда True оно замедляется где то в 5 раз) 
        temp_svm = SVC(probability=False, kernel='linear', random_state=21, C=c)
        for train_idx, valid_idx in skf.split(X, y):
            X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
            y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
            
            temp_svm.fit(X_train_fold, y_train_fold)
        
            train_acc = temp_svm.score(X_train_fold, y_train_fold)
            valid_acc = temp_svm.score(X_valid_fold, y_valid_fold)
        
            valid_scores_temp.append(valid_acc)

        if np.mean(valid_scores_temp) > best_acc:
            best_acc = np.mean(valid_scores_temp)
            best_C = c 
        
print(f"Best average accuracy on crossval is: {best_acc:.5f}")
print(f"Best C is: {best_C}")

Best average accuracy on crossval is: 0.76158
Best C is: 1000


## 4. Дерево

### а. Регуляризация по умолчанию
Обучите базовую модель, используя только один параметр max_depth=10 и random_state=21.
Для оценки точности модели используйте стратифицированную K-кратную перекрестную проверку с разбиением выборки 10.
Формат результата выполнения кода, в котором вы обучали и оценивали базовую модель, должен быть аналогичен формату, который вы получили для logreg.

In [80]:
tree = DecisionTreeClassifier(max_depth=10, random_state=21)
skf = StratifiedKFold(n_splits=10, random_state=21, shuffle=True)
valid_scores = []
for train_idx, valid_idx in skf.split(X, y):
    train_fold, valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
    y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
    
    tree.fit(train_fold, y_train_fold)
    
    train_acc = tree.score(train_fold, y_train_fold)
    valid_acc = tree.score(valid_fold, y_valid_fold)
    
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')
print(f'Std is {np.std(valid_scores):.5f}')


train -  0.82004   |   valid -  0.79290
train -  0.82663   |   valid -  0.69822
train -  0.82927   |   valid -  0.76331
train -  0.81806   |   valid -  0.71598
train -  0.82268   |   valid -  0.74556
train -  0.80554   |   valid -  0.77515
train -  0.83333   |   valid -  0.75595
train -  0.81555   |   valid -  0.76786
train -  0.81225   |   valid -  0.77381
train -  0.81752   |   valid -  0.69048
Average accuracy on crossval is 0.74792
Std is 0.03306


### б. Оптимизация параметров регуляризации
1) В ячейках ниже попробуйте разные значения параметра max_depth.
2) В качестве бонуса поэкспериментируйте с другими параметрами регуляризации, пытаясь найти наилучшую комбинацию.

In [81]:
param_combinations = [{"max_depth": 3, "min_samples_leaf":1},
                      {"max_depth": 3, "min_samples_leaf":2},
                      {"max_depth": 3, "min_samples_leaf":3},
                      {"max_depth": 5, "min_samples_leaf":1},
                      {"max_depth": 5, "min_samples_leaf":2},
                      {"max_depth": 5, "min_samples_leaf":3},
                      {"max_depth": 7, "min_samples_leaf":1},
                      {"max_depth": 7, "min_samples_leaf":2},
                      {"max_depth": 7, "min_samples_leaf":3},
                      {"max_depth": 15, "min_samples_leaf":1},
                      {"max_depth": 15, "min_samples_leaf":2},
                      {"max_depth": 15, "min_samples_leaf":3},
                      {"max_depth": 30, "min_samples_leaf":1},
                      {"max_depth": 30, "min_samples_leaf":2},
                      {"max_depth": 30, "min_samples_leaf":3},
                      ]
best_acc = 0
best_combination=''
for params in param_combinations:
    for train_idx, valid_idx in skf.split(X, y):
        tempTree = DecisionTreeClassifier(max_depth=params['max_depth'], min_samples_leaf=params['min_samples_leaf'], random_state=21)
        X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
        
        tempTree.fit(X_train_fold, y_train_fold)
        valid_acc = tempTree.score(X_valid_fold, y_valid_fold)
        if valid_acc > best_acc:
            best_acc = valid_acc
            best_combination = params
        
print(f"Best average accuracy on crossval is {best_acc:.5f}")
print(f"Best combination is: {best_combination}")

Best average accuracy on crossval is 0.94083
Best combination is: {'max_depth': 30, 'min_samples_leaf': 1}


## 5. Случайный лес
### а. Регуляризация по умолчанию
1) Обучите базовую модель, используя только параметры n_estimators=50, max_depth=14, random_state=21.
2) 10 Для оценки точности модели используйте стратифицированную K-кратную перекрестную проверку с разбиением выборки.
3) Формат результата выполнения кода, в котором вы обучали и оценивали базовую модель, должен быть аналогичен формату, который вы получили для logreg.

In [82]:
rndForest = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)

valid_scores = []
for train_idx, valid_idx in skf.split(X, y):
    X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
    y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
    
    rndForest.fit(X_train_fold, y_train_fold)
    train_acc = rndForest.score(X_train_fold, y_train_fold)
    valid_acc = rndForest.score(X_valid_fold, y_valid_fold)
    
    valid_scores.append(valid_acc)
    
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Average accuracy on crossval is {np.mean(valid_scores):.5f}')
print(f'Std is {np.std(valid_scores):.5f}')
    

train -  0.97034   |   valid -  0.90533
train -  0.96704   |   valid -  0.87574
train -  0.96902   |   valid -  0.91124
train -  0.97429   |   valid -  0.89349
train -  0.96243   |   valid -  0.86982
train -  0.96638   |   valid -  0.94083
train -  0.97036   |   valid -  0.92262
train -  0.97036   |   valid -  0.91667
train -  0.96838   |   valid -  0.89881
train -  0.97563   |   valid -  0.88690
Average accuracy on crossval is 0.90214
Std is 0.02069


### б. Оптимизация параметров регуляризации
1) В новых ячейках попробуйте разные значения параметров max_depth и n_estimators.
2) В качестве бонуса поэкспериментируйте с другими параметрами регуляризации, пытаясь найти наилучшую комбинацию.

In [83]:
param_combinations = [{"max_depth": 3, "n_estimators":25},
                      {"max_depth": 3, "n_estimators":50},
                      {"max_depth": 3, "n_estimators":100},
                      {"max_depth": 3, "n_estimators":200},
                      {"max_depth": 25, "n_estimators":25},
                      {"max_depth": 25, "n_estimators":50},
                      {"max_depth": 25, "n_estimators":100},
                      {"max_depth": 25, "n_estimators":200},
                      {"max_depth": 50, "n_estimators":25},
                      {"max_depth": 50, "n_estimators":50},
                      {"max_depth": 50, "n_estimators":100},
                      {"max_depth": 50, "n_estimators":200},
                      ]
best_acc = 0
best_combination=''
for params in param_combinations:
    for train_idx, valid_idx in skf.split(X, y):
        tempForest = RandomForestClassifier(max_depth=params['max_depth'], min_samples_leaf=params['n_estimators'], random_state=21)
        X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
        
        tempForest.fit(X_train_fold, y_train_fold)
        valid_acc = tempForest.score(X_valid_fold, y_valid_fold)
        if valid_acc > best_acc:
            best_acc = valid_acc
            best_combination = params
        
print(f"Best average accuracy on crossval is {best_acc:.5f}")
print(f"Best combination is: {best_combination}")

Best average accuracy on crossval is 0.55357
Best combination is: {'max_depth': 25, 'n_estimators': 25}


## 6. Прогнозы
1) Выберите лучшую модель и используйте её для прогнозирования на тестовом наборе данных.
2) Рассчитайте окончательную точность.
3) Анализ: для какого дня недели ваша модель допускает наибольшее количество ошибок (в процентах от общего числа образцов этого класса в вашем тестовом наборе данных).
4) Сохраните модель.



In [84]:
y_pred = rndForest.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on test set is {acc:.5f}")
dayofweekAnalyze = {"0":0,"1":0,"2":0,"3":0,"4":0,"5":0,"6":0}

for i in range(len(y_pred)):
    if y_pred[i]!=y_test.iloc[i]:
        dayofweekAnalyze[str(y_test.iloc[i])]+=1
analyze ="Day of the week with the most errors: "
maxError = 0
for i in range(len(dayofweekAnalyze)):
    if dayofweekAnalyze[str(i)]>maxError:
        maxError = dayofweekAnalyze[str(i)]
        
for i in range(len(dayofweekAnalyze)):
    if dayofweekAnalyze[str(i)]==maxError:
        analyze+= str(i)+","
analyze = analyze[:len(analyze)-1]
print(analyze)

Accuracy on test set is 0.97337
Day of the week with the most errors: 0,2,3


In [85]:
joblib.dump(rndForest, '../models/rndForest.joblib')

['../models/rndForest.joblib']